# Data Preprocessing Notebook (Decision Tree)
## CS 171 Final Project: West Coast Swing Dance Pattern Classification

**Author:** Nguyen Pham

This notebook handles data preprocessing for the Decision Tree model:
- Download YouTube dance videos
- Extract video frames
- Extract MediaPipe pose keypoints
- Prepare data for trick classification (Sugar Push vs Sugar Tag)

**Imports**

In [ ]:
import os
from pathlib import Path
import pandas as pd
import yt_dlp
import re
from urllib.parse import urlparse, parse_qs
import sys

sys.path.insert(0, str(Path.cwd().parent))
from scripts.extract_frames import extract_frames
from scripts.extract_keypoints import extract_keypoints

**Download Videos**

In [ ]:
def download_video(url, name, output_path, start_time=0, duration=4):
    # Combine output path with filename
    output_path = output_path / f'{name}.mp4'
    
    ydl_opts = {
        'format': 'best',
        'outtmpl': str(output_path),
        'quiet': True,
        'no_warnings': True,
        'download_ranges': lambda info_dict, ydl: [
            {
                'start_time': start_time,
                'end_time': start_time + duration,
            }
        ],
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
            print(f"Downloaded video: {name}")
        return True
    except Exception as e:
        print(f"Error downloading video: {e}")
        return False

**Extract Youtube ID**

In [ ]:
def extract_youtube_id(url):
    if not url:
        return None
    
    # Parse URL
    parsed = urlparse(url)

    patterns = [
        r'(?:youtube\.com\/shorts\/)([a-zA-Z0-9_-]{11})', # www.youtube.com/shorts/VIDEO_ID
        r'(?:youtube\.com\/embed\/)([a-zA-Z0-9_-]{11})', # www.youtube.com/embed/VIDEO_ID
        r'(?:youtube\.com\/v\/)([a-zA-Z0-9_-]{11})', # www.youtube.com/v/VIDEO_ID
        r'(?:youtu\.be\/)([a-zA-Z0-9_-]{11})', # youtu.be/VIDEO_ID
        r'(?:youtube\.com\/watch\?v=)([a-zA-Z0-9_-]{11})', # www.youtube.com/watch?v=VIDEO_ID
        r'(?:youtube\.com\/)([a-zA-Z0-9_-]{11})', # www.youtube.com/VIDEO_ID
    ]

    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            print("Found video ID:", match.group(1))
            return match.group(1)

    print("No video ID found in URL:", url)
    return None
    

**Create Labels**

In [ ]:
def add_label(csv_path, url,start_time=0, duration=10, action_class=None, dance_style=None, labels=None, division=None, pattern=None):
    # Check if the CSV file exists
    if not os.path.exists(csv_path):
        df = pd.DataFrame(columns=['id', 'youtube_id', 'start_time', 'duration', 'action_class', 'dance_style','labels', 'division', 'pattern'])
        df.to_csv(csv_path, index=False)
    else:
        # Load existing CSV
        df = pd.read_csv(csv_path)

    # Get next ID
    if len(df) == 0:
        next_id = 1
    else:
        next_id = df['id'].max() + 1

    # Extract ID from URL
    video_id = extract_youtube_id(url)
    
    # Add new rows
    new_row = {
        'id': next_id,
        'youtube_id': video_id,
        'start_time': start_time,
        'duration': duration,
        'action_class': action_class,
        'dance_style': dance_style,
        'labels': labels,
        'division': division,
        'pattern': pattern,
    }

    # Append new rows
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    # Save updated CSV
    df.to_csv(csv_path, index=False)
    print(f"Added video with ID {next_id} to {csv_path}")

In [ ]:
os.makedirs('../data', exist_ok=True)
CSV_PATH = '../data/hp.csv'

**Batch Download Function**

In [ ]:
def download_all_from_csv(csv_path, folder):
    df = pd.read_csv(csv_path)

    for index, row in df.iterrows():
        # Create folder with skill level (division)
        os.makedirs(f'../data/raw/{folder}/{row["division"]}', exist_ok=True)

        # Download video
        video_name = f"{row['division']}_{row['id']}"
        VIDEO_DIR = Path(f'../data/raw/{folder}/{row["division"]}')
        
        print(f"Downloading {row['id']}: {row['division']}...")
        download_video(
            url=row['youtube_id'],
            name=video_name,
            output_path=VIDEO_DIR,
            start_time=row['start_time'],
            duration=row['duration'],
        )

In [ ]:
# Download all videos
CSV_PATH = '../data/hp.csv'
download_all_from_csv(CSV_PATH, "videos_1")

In [ ]:
extract_frames(input_dir="../data/raw/videos_1",output_dir="../data/frames_1")
extract_keypoints(input_dir="../data/frames_1", output_dir="../data/keypoints_1", static_mode=True)